# Circuit registration

Copyright (c) 2026 Open Brain Institute

Authors: Christoph Pokorny

Last modified: 06.2026

## Summary
This notebook allows a user to register a new SONATA circuit together with metadata as a private `Circuit` entity in the entitycore database. Additional circuit assets (connectivity matrix, basic connectivity plots, etc.) will be generated automatically upon upload.

## Use
The SONATA circuit should be provided by compressing the whole circuit folder into a zip file and copying it into the JupyterHub workspace. All required metadata can be entered diretly within the notebook. Where applicable, interactive widgets can be used to select metadata options and linked entities in a user-friendly way. All required linked entities (publications, contributors, subjects, etc.) must exist already in the database, i.e., they have to be registered separately beforehand.

Optionally, the main overview and simulation designer images can be provided as well; otherwise default images will be generated.

**Important:** [SNAP circuit validation](https://github.com/openbraininstitute/snap#circuit-validation) needs to pass for a new circuit to be registered.


In [ ]:
from entitysdk import Client, models
from entitysdk.types import CircuitBuildCategory, CircuitScale, DerivationType, TargetSimulator
from obi_auth import get_token
from obi_notebook import get_projects
from obi_notebook.get_environment import get_environment
from obi_one.utils.circuit_registration import register_circuit_from_metadata
from pathlib import Path

In [ ]:
# Disable info logging (optional)
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

## Project selection

As a first step, we select the project under which to register the new circuit as a private `Circuit` entity.


In [ ]:
# environment = get_environment()
environment = "staging"
token = get_token(environment=environment, auth_mode="daf")
project_context = get_projects.get_projects(token, env=environment)

Then, we get the database client for accessing entitycore.

In [ ]:
client = Client(environment=environment, project_context=project_context, token_manager=token)

## Circuit information

### General metadata

**IMPORTANT:**
- **The circuit name must not yet exist in entitycore (to avoid duplicates)**

- **Linked entities must exist in entitycore. Otherwise, they must first be registered:**
  - Species
  - Subject
  - Brain region hierarchy
  - Brain region
  - License


In [ ]:
print("Collecting metadata options...")

# Query available species from entitycore
print("> Querying species")
all_species = client.search_entity(entity_type=models.Species).all()
species_names = sorted([s.name for s in all_species])
default_species = "Rattus norvegicus"
if default_species not in species_names:
    default_species = species_names[0]

# Query available circuits (exclude single-neuron scale)
print("> Querying circuits")
circuit_scales = [scale.name for scale in CircuitScale if scale != "single"]
all_circuits = client.search_entity(entity_type=models.Circuit, query={"scale__in": circuit_scales}).all()
circuit_names = [(f"{c.scale}: {c.name}{' [public]' if c.authorized_public else ''}", c.name) for c in all_circuits]
circuit_names = sorted(circuit_names, key=lambda c: c[0])

# Query available subjects from entitycore
print("> Querying subjects")
all_subjects = client.search_entity(entity_type=models.Subject).all()
subject_species = set(s.species for s in all_subjects)
subject_names = {species.name: sorted([s.name for s in all_subjects if s.name != "Unknown" and s.species == species]) for species in subject_species}

# Query available brain region hierarchies from entitycore
print("> Querying brain region hierarchies")
all_hierarchies = client.search_entity(entity_type=models.BrainRegionHierarchy).all()
hierarchy_species = set(h.species for h in all_hierarchies)
hierarchy_names = {species.name: sorted([h.name for h in all_hierarchies if h.species == species]) for species in hierarchy_species}

# Query available brain regions from entitycore (grouped by hierarchy)
print("> Querying brain regions")
all_brain_regions = client.search_entity(entity_type=models.BrainRegion).all()
brain_region_names = {h.name: sorted([r.name for r in all_brain_regions if r.hierarchy_id == h.id]) for h in all_hierarchies}

# Query available licenses from entitycore
print("> Querying licenses")
all_licenses = client.search_entity(entity_type=models.License).all()
license_names = sorted([l.label for l in all_licenses])

In [ ]:
import ipywidgets as widgets
from IPython.display import display


# --- Required fields ---
w_name = widgets.Text(
    value="",
    placeholder="e.g. EXAMPLE_01__N_10__top_nodes_dim6",
    description="Name:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

w_description = widgets.Textarea(
    value="",
    placeholder="e.g. Example circuit with 10 neurons",
    description="Description:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%", height="60px"),
)

w_build_category = widgets.Dropdown(
    options=[(e.name, e) for e in CircuitBuildCategory],
    description="Build category:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

w_species = widgets.Dropdown(
    options=species_names,
    value=default_species,
    description="Species:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

w_subject = widgets.Dropdown(
    options=subject_names.get(default_species, []),
    description="Subject:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

def _update_subjects(change):
    w_subject.options = subject_names.get(change["new"], [])

w_species.observe(_update_subjects, names="value")

w_brain_region_hierarchy = widgets.Dropdown(
    options=hierarchy_names.get(default_species, []),
    description="Brain region hierarchy:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

def _update_hierarchies(change):
    w_brain_region_hierarchy.options = hierarchy_names.get(change["new"], [])

w_species.observe(_update_hierarchies, names="value")

w_brain_region = widgets.Dropdown(
    options=brain_region_names.get(w_brain_region_hierarchy.value, []),
    description="Brain region:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

def _update_brain_regions(change):
    w_brain_region.options = brain_region_names.get(change["new"], [])

w_brain_region_hierarchy.observe(_update_brain_regions, names="value")

# --- Optional fields ---
w_target_simulator = widgets.Dropdown(
    options=[("None (use default)", None)] + [(e.name, e) for e in TargetSimulator],
    value=None,
    description="Target simulator:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

w_root = widgets.Dropdown(
    options=[("None", None)] + circuit_names,
    value=None,
    description="Root circuit:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

w_parent = widgets.Dropdown(
    options=[("None", None)] + circuit_names,
    value=None,
    description="Parent circuit:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

w_derivation_type = widgets.Dropdown(
    options=[(e.name, e) for e in DerivationType if e.name in ("circuit_extraction", "circuit_rewiring")],
    description="Derivation type:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
    disabled=True,
)

def _toggle_derivation(change):
    w_derivation_type.disabled = change["new"] is None

w_parent.observe(_toggle_derivation, names="value")

w_license = widgets.Dropdown(
    options=[("None", None)] + [(n, n) for n in license_names],
    value=None,
    description="License:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

w_published_in = widgets.Text(
    value="",
    placeholder="e.g. Reimann et al and Isbister et al",
    description="Published in:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

w_contact = widgets.Text(
    value="",
    placeholder="e.g. support@openbraininstitute.org",
    description="Contact e-mail:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="80%"),
)

w_experiment_date = widgets.DatePicker(
    description="Experiment date:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="50%"),
)

# --- Layout ---
required_box = widgets.VBox(
    [
        widgets.HTML("<h4>Required</h4>"),
        w_name,
        w_description,
        w_build_category,
        w_species,
        w_subject,
        w_brain_region_hierarchy,
        w_brain_region,
    ]
)

optional_box = widgets.VBox(
    [
        widgets.HTML("<h4>Optional</h4>"),
        w_target_simulator,
        w_root,
        w_parent,
        w_derivation_type,
        w_license,
        w_published_in,
        w_contact,
        w_experiment_date,
    ]
)

display(required_box, optional_box)

Run the cell below to assemble the `circuit_metadata` dict from the widget values above.

In [ ]:
circuit_metadata = {
    "name": w_name.value,
    "description": w_description.value,
    "build_category": w_build_category.value,
    "species": w_species.value,
    "subject": w_subject.value,
    "brain_region_hierarchy": w_brain_region_hierarchy.value,
    "brain_region": w_brain_region.value,
    "target_simulator": w_target_simulator.value,
    "root": w_root.value or None,
    "parent": w_parent.value or None,
    "derivation_type": w_derivation_type.value if not w_derivation_type.disabled else None,
    "license": w_license.value or None,
    "published_in": w_published_in.value or None,
    "contact": w_contact.value or None,
    "experiment_date": w_experiment_date.value.strftime("%d.%m.%Y") if w_experiment_date.value else None,
}

# Quick validation of required fields
_required = ["name", "description", "build_category", "species", "subject", "brain_region_hierarchy", "brain_region"]
_missing = [k for k in _required if not circuit_metadata.get(k)]
if circuit_metadata.get("parent") and not circuit_metadata.get("derivation_type"):
    _missing.append("derivation_type")
if _missing:
    raise ValueError(f"Missing required metadata fields: {_missing}")

# Validate circuit name
if circuit_metadata["name"] in [c.name for c in all_circuits]:
    raise ValueError(f"Circuit name '{circuit_metadata['name']}' already exists!")

print("circuit_metadata:")
for k, v in circuit_metadata.items():
    print(f"  {k}: {v!r}")
